In [ ]:
import os
import sys
import random
import unicodedata
import contextlib
import io
from pathlib import Path

# Ensure CWD is the repo root (needed for naibbe_v2's CSV import)
REPO_ROOT = Path.cwd()
os.chdir(str(REPO_ROOT))

# Import cipher modules (suppress naibbe_v2's import-time output)
with contextlib.redirect_stdout(io.StringIO()):
    import naibbe_habit

import naibbe_v2
import voynichesque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parameters (mirrored from figure_utils/rmsf/rmsf.ipynb)
MAX_BITS = 500_000
WINDOW_SIZES = np.unique(
    np.logspace(1.3, np.log10(MAX_BITS // 10), num=20, dtype=int)
)
MAX_LAG = 1000

# Paths (all relative to repo root)
INPUT_PATH = REPO_ROOT / "input" / "examples" / "nathist_book16.txt"
CURRIER_B_PATH = (
    REPO_ROOT / "figure_utils" / "gaskell_bowern_2022" / "data" /
    "voynichese" / "Voynichese - Currier B - EVA Basic.txt"
)
VOYNICH_B_HURST_CSV = (
    REPO_ROOT / "figure_utils" / "rmsf" / "long_range_correlation_results.csv"
)
VOYNICH_B_RMSF_CSV = (
    REPO_ROOT / "figure_utils" / "rmsf" / "long_range_plots" /
    "voynich_b_rmsf_data.csv"
)
OUTPUT_DIR = REPO_ROOT / "figure_utils" / "rmsf" / "long_range_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# === RMSF/Hurst analysis (ported from figure_utils/rmsf/rmsf.ipynb) ===

def text_to_binary(text: str) -> str:
    """Encode unicode text into a 5-bit-per-letter binary string."""
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if c.isalpha()).lower()
    alpha = "abcdefghijklmnopqrstuvwxyz"
    rank_map = {char: format(i + 1, "05b") for i, char in enumerate(alpha)}
    return "".join(rank_map.get(c, "") for c in text)

def binary_to_walk(binary_str: str) -> np.ndarray:
    """Map a binary string to a cumulative ±1 walk."""
    return np.cumsum([1 if bit == "1" else -1 for bit in binary_str])

def compute_rms_fluctuation(walk: np.ndarray, window_sizes: np.ndarray) -> np.ndarray:
    """Compute F(n) for a set of window sizes."""
    n = len(walk)
    f_values = []
    for l in window_sizes:
        if n - l <= 1:
            f_values.append(0)
            continue
        dy = walk[l:] - walk[:-l]
        var = np.mean(dy ** 2) - np.mean(dy) ** 2
        f_values.append(float(np.sqrt(max(var, 0))))
    return np.array(f_values)

def compute_autocorrelation(bit_array: np.ndarray, max_lag: int) -> tuple[np.ndarray, np.ndarray]:
    """Compute lag-l and cumulative autocorrelations."""
    n = len(bit_array)
    mean_val = float(np.mean(bit_array))
    c = []
    for l in range(1, max_lag + 1):
        cov = float(np.mean(bit_array[l:] * bit_array[:-l])) - mean_val ** 2
        c.append(cov)
    c_arr = np.array(c)
    c_cum = np.cumsum(c_arr) / np.arange(1, max_lag + 1)
    return c_arr, c_cum

def fit_hurst(window_sizes: np.ndarray, f_values: np.ndarray) -> tuple[float, int]:
    """Estimate Hurst exponent and crossover point from F(n)."""
    log_n = np.log10(window_sizes)
    log_f = np.log10(f_values)
    hurst = round(float(np.polyfit(log_n, log_f, 1)[0]), 3)
    diffs = np.diff(log_f / log_n)
    crossover_idx = int(np.argmax(diffs))
    crossover = int(window_sizes[crossover_idx]) if crossover_idx < len(window_sizes) else -1
    return hurst, crossover

def analyze_ciphertext(ct_string: str) -> dict:
    """Run the full RMSF + Hurst + autocorrelation pipeline."""
    binary = text_to_binary(ct_string)[:MAX_BITS]
    bit_array = np.array([int(b) for b in binary], dtype=np.int8)
    walk = binary_to_walk(binary)
    f_values = compute_rms_fluctuation(walk, WINDOW_SIZES)
    c, c_cum = compute_autocorrelation(bit_array, MAX_LAG)
    hurst, crossover = fit_hurst(WINDOW_SIZES, f_values)
    return {
        "bit_array": bit_array, "walk": walk, "f_values": f_values,
        "c": c, "c_cum": c_cum, "hurst": hurst, "crossover_n": crossover,
    }


In [ ]:
# === Encryption configurations ===

# Habit config: (label, bifolia_per_quire, tokens_per_page, p_habit)
HABIT_CONFIGS = [
    ("habit_control", 4, 160, 0.0),  # same pipeline, no table habit
    ("habit_03", 4, 160, 0.3),
    ("habit_05", 4, 160, 0.5),
    ("habit_10", 4, 160, 1.0),
]

SEED = 42
print("Exploratory single-seed run; do not interpret as reliability evidence.")

def encrypt_vanilla(seed=SEED):
    random.seed(seed)
    all_tokens = []
    with open(str(INPUT_PATH), "r", encoding="utf-8") as fin:
        for line in fin:
            cleaned = naibbe_v2.clean_line(line)
            if not cleaned:
                continue
            tokens = naibbe_v2.encrypt_naibbe(
                cleaned, naibbe_v2.naibbe_tables,
                naibbe_v2.placeholder_to_glyph,
                use_78=naibbe_v2.USE_78_CARD_DECK,
            )
            all_tokens.extend(tokens)
    return " ".join(all_tokens)

def encrypt_habit(bpf, tpp, p_habit, seed=SEED):
    rng = random.Random(seed)
    all_tokens = []
    for cleaned, tokens, _ in naibbe_habit.iter_encrypted_lines(
        str(INPUT_PATH), naibbe_habit.USE_78_CARD_DECK,
        bpf, tpp, p_habit, rng,
    ):
        if not cleaned:
            continue
        all_tokens.extend(tokens)
    return " ".join(all_tokens)


In [ ]:
# === Encrypt all configurations ===
ciphertexts = {}
analyses = {}

print("VANILLA ENCRYPTION (baseline)")
print("-" * 60)
ct = encrypt_vanilla()
ciphertexts["vanilla"] = ct
analyses["vanilla"] = analyze_ciphertext(ct)
a = analyses["vanilla"]
print(f"  vanilla       \u2192 H={a['hurst']:.3f}  crossover\u2248{a['crossover_n']}  "
      f"({len(ct.split())} tokens)")

print()
print("HABIT ENCRYPTION (bifolium favourite-table model)")
print("-" * 60)
for label, bpf, tpp, p_habit in HABIT_CONFIGS:
    ct = encrypt_habit(bpf, tpp, p_habit)
    ciphertexts[label] = ct
    analyses[label] = analyze_ciphertext(ct)
    a = analyses[label]
    print(f"  {label:14s}  bifolia={bpf}  p_habit={p_habit:.1f}  "
          f"\u2192 H={a['hurst']:.3f}  "
          f"crossover\u2248{a['crossover_n']}  ({len(ct.split())} tokens)")


In [ ]:
# === Voynich B reference ===
with open(str(CURRIER_B_PATH), "r", encoding="utf-8") as f:
    currier_b_text = " ".join(line.strip() for line in f if line.strip())
currier_b_analysis = analyze_ciphertext(currier_b_text)
currier_b_metrics = voynichesque.summarize_ciphertext(currier_b_text)

# Reference Hurst from the long-range correlation results
lr_df = pd.read_csv(str(VOYNICH_B_HURST_CSV))
voynich_b_row = lr_df[lr_df["Text"] == "voynich_b"]
voynich_b_hurst = float(voynich_b_row["Hurst Exponent"].iloc[0])
voynich_b_crossover = int(voynich_b_row["Crossover Point"].iloc[0])

print(f"Voynich B reference: H={voynich_b_hurst:.3f}, "
      f"crossover={voynich_b_crossover}")
print(f"  TTR={currier_b_metrics['ttr']:.3f}, "
      f"char_H={currier_b_metrics['char_entropy']:.3f}, "
      f"cond_H={currier_b_metrics['cond_char_entropy']:.3f}")


In [ ]:
# === Voynichesque metrics for all configs ===
metrics_rows = []
for label, ct in sorted(ciphertexts.items()):
    m = voynichesque.summarize_ciphertext(ct)
    m["config"] = label
    m["seed"] = SEED
    m["hurst"] = analyses[label]["hurst"]
    metrics_rows.append(m)
currier_b_metrics_copy = dict(currier_b_metrics)
currier_b_metrics_copy["config"] = "currier_b"
currier_b_metrics_copy["seed"] = None
currier_b_metrics_copy["hurst"] = voynich_b_hurst
metrics_rows.append(currier_b_metrics_copy)

metrics_df = pd.DataFrame(metrics_rows)[
    ["config", "seed", "n_tokens", "n_types", "ttr", "mean_token_len",
     "char_entropy", "cond_char_entropy", "hurst"]
]
metrics_df.to_csv(OUTPUT_DIR.parent / "ssc_metrics_comparison.csv", index=False)
print(metrics_df.to_string(index=False))


In [ ]:
# === Hurst comparison table (computed, not hardcoded) ===
hurst_rows = []
a = analyses["vanilla"]
hurst_rows.append({
    "config": "vanilla", "model": "baseline",
    "seed": SEED,
    "hurst": a["hurst"], "crossover_point": a["crossover_n"],
})
for label, bpf, tpp, p_habit in HABIT_CONFIGS:
    a = analyses[label]
    hurst_rows.append({
        "config": label, "model": "habit",
        "seed": SEED,
        "bifolia_per_quire": bpf, "tokens_per_page": tpp,
        "p_habit": p_habit,
        "hurst": a["hurst"], "crossover_point": a["crossover_n"],
    })
hurst_rows.append({
    "config": "voynich_b", "model": "reference",
    "seed": None,
    "hurst": voynich_b_hurst, "crossover_point": voynich_b_crossover,
})
hurst_df = pd.DataFrame(hurst_rows)
hurst_df.to_csv(OUTPUT_DIR.parent / "ssc_hurst_comparison.csv", index=False)
print(hurst_df.to_string(index=False))


In [ ]:
# === Closest config to Voynich B ===
all_rows = [r for r in hurst_rows if r["config"] != "voynich_b"]
closest = min(all_rows, key=lambda r: abs(r["hurst"] - voynich_b_hurst))
if closest["model"] == "habit":
    params = (f"bpf={closest['bifolia_per_quire']}, "
              f"p_habit={closest['p_habit']}")
else:
    params = ""
print(f"Voynich B reference: H={voynich_b_hurst:.3f}")
print(f"Closest config in this exploratory run: '{closest['config']}' "
      f"(H={closest['hurst']:.3f}, {params})")
print(f"Gap: {abs(closest['hurst'] - voynich_b_hurst):.3f}")


In [ ]:
# === RMSF overlay plot ===
fig, ax = plt.subplots(figsize=(9, 7))

# Voynich B reference curve
ref = pd.read_csv(str(VOYNICH_B_RMSF_CSV))
log_n_ref = np.log10(ref["window_size"].to_numpy())
log_f_ref = np.log10(ref["rmsf"].to_numpy())
log_n = np.log10(WINDOW_SIZES)

# Reference slope = 0.5 line
slope_ref = log_f_ref[0] + 0.5 * (log_n_ref - log_n_ref[0])
ax.plot(log_n_ref, slope_ref, "k--", linewidth=1.0, label="Reference slope = 0.5")
ax.plot(log_n_ref, log_f_ref, "k-", linewidth=3.0, label="Voynich Currier B")

# Vanilla config (baseline)
a = analyses["vanilla"]
ax.plot(log_n, np.log10(a["f_values"]), "-", linewidth=1.2,
        label=f"vanilla (H\u2248{a['hurst']:.3f})")

# Habit configs (dashed lines)
for label, *_ in HABIT_CONFIGS:
    a = analyses[label]
    ax.plot(log_n, np.log10(a["f_values"]), "--", linewidth=1.2,
            label=f"{label} (H\u2248{a['hurst']:.3f})")

ax.set_xlabel("log10(window size n)")
ax.set_ylabel("log10(F(n))")
ax.set_title("Habit & Vanilla vs Voynich Currier B \u2014 RMSF Comparison")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ssc_vs_voynich_b_overlay.png", dpi=120)
plt.show()


In [ ]:
# === Autocorrelation plot ===
fig, ax = plt.subplots(figsize=(9, 6))
lags = np.arange(1, MAX_LAG + 1)
for label in ["vanilla"] + [l for l, *_ in HABIT_CONFIGS]:
    a = analyses[label]
    style = "-" if label == "vanilla" else "--"
    ax.plot(lags, a["c_cum"], style, linewidth=0.8, label=label)
ax.set_xlabel("Lag \u2113")
ax.set_ylabel("C_c(\u2113)")
ax.set_title("Cumulative Step Autocorrelation \u2014 Habit & Vanilla")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="best")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ssc_autocorrelation_comparison.png", dpi=120)
plt.show()
